# Modeling

Imports

In [1]:
import re
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    SelectFromModel,
    mutual_info_classif
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score
)
from sklearn.model_selection import (
    TimeSeriesSplit,
    train_test_split
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier, callback
import shap
import joblib

Load WTI Data

In [2]:
wti = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet")

Load Tweet Data

In [3]:
tweets = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

In [4]:
tweets.shape

(15242, 53)

In [5]:
wti.shape

(4206118, 77)

## Train-Validation-Test-Split

In [6]:
tweets_pre2020 = tweets[tweets['timestamp_utc'] < '2020-01-01'] # wird für training und validation verwendet
tweets_post2020 = tweets[tweets['timestamp_utc'] >= '2020-01-01'] # als test set für die out of sample performance

## ML-Pipeline

### 1. Config

**Hinweis: Die folgenden Konfigurationen dienen nur einem Testlauf, um die Funktionsweise der Pipeline zu verstehen. Das Training für die finalen Projekt-Ergebnisse hat mehrere Stunden gedauert! Die originalen Konfigurationen sind darunter in auskommentierter Form zu finden**

In [7]:
TIME_WTI = "timestamp_utc"
TIME_TWEET = "timestamp_utc"


CONFIG = {
    "outer_splits": 2,
    "inner_splits": 2,
    "gap": 0,

    "targets": ["y_up_2"],  # nur ein Target

    "feature_sets": ["enhanced"],

    "models": ["logreg"],

    "optuna_trials": 1,

    "embedding_pca_components": 30,
}

PROD Configs - die originalen Konfigurationen

In [8]:
# [c for c in wti.columns if c.startswith("y_")]

In [9]:
# # PCA_COMPONENTS = 20
# # N_SPLITS = 5
# # PURGE_MINUTES = 1440

# TIME_WTI = "timestamp_utc"
# TIME_TWEET = "timestamp_utc"


# CONFIG = {
#     "outer_splits": 5,
#     "inner_splits": 3,
#     "gap": 150,

#     "targets": [
#         'y_up_2',
#         'y_up_5',
#         'y_up_15',
#         'y_up_30',
#         'y_up_60',
#         'y_up_240',
#         'y_up_720',
#         'y_up_1440'
#     ],
#     "feature_sets": ["baseline", "enhanced"],
#     "models": ["logreg", "xgboost", "mlp"],

#     "optuna_trials": 30,

#     # "top_k_features": 0.7,

#     # PCA for embeddings
#     "embedding_pca_components": 30,

#     # Mutual Information
#    #  "mi_top_k": 150,    
# }

### 2. Embedding Parser

In [10]:
def parse_embedding(x):
    return np.fromstring(x.strip("[]"), sep=" ")

### 3. Tweet Preprocessing + Embeddings

In [11]:
def prepare_tweets(df):

    df = df.copy()

    df[TIME_TWEET] = pd.to_datetime(df[TIME_TWEET], utc=True)

    df["embedding_vec"] = df["embedding"].apply(parse_embedding)

    emb = np.vstack(df["embedding_vec"].values)

    emb_cols = [f"emb_{i}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    df = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

    return df

### 4. Tweet Aggregation

Da es zum Teil mehrere Tweets innerhalb einer Minute gibt

In [12]:
def aggregate_tweets(df):

    df = df.copy()

    df["minute"] = df[TIME_TWEET].dt.floor("min")

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    agg_dict = {

        # =========================
        # FinBERT
        # =========================
        "finbert_positive": "mean",
        "finbert_neutral": "mean",
        "finbert_negative": "mean",
        "finbert_sentiment_score": "mean",
        "finbert_entropy": "mean",
        "finbert_confidence": "max",
        "finbert_polarization": "max",

        # =========================
        # Financial NLP Scores
        # =========================
        "financial_uncertainty_score": "max",
        "financial_risk_sentiment": "mean",

        # =========================
        # Text Style
        # =========================
        "caps_ratio": "mean",
        "exclamation_count": "max",
        "exclamation_count_log": "max",

        "tweet_length": "mean",
        "tweet_length_log": "mean",

        "market_aggression_index": "max",

        # =========================
        # Tweet Dynamics
        # =========================
        "time_since_last_tweet_min": "min",

        "rolling_tweet_frequency_60m": "max",
        "rolling_tweet_frequency_6h": "max",
        "rolling_tweet_frequency_24h": "max",

        "tweet_burst_indicator": "max",
        "tweet_acceleration_6h": "max",

        # =========================
        # Sentiment Dynamics
        # =========================
        "sentiment_delta_vs_previous": "mean",
        "rolling_sentiment_mean_60m": "mean",
        "rolling_sentiment_std_60m": "mean",

        # =========================
        # Presidency Regime
        # =========================
        "is_trump_president": "max",

        # =========================
        # Oil / Macro Scores
        # =========================
        "wti_bullish_score": "max",
        "wti_bearish_score": "max",

        "energy_supply_score": "max",
        "geopolitical_oil_risk_score": "max",
        "usd_strength_pressure_score": "max",
        "fed_monetary_policy_score": "max",
        "risk_sentiment_score": "max",

        "trump_energy_policy_score": "max",
        "trump_market_shock_language_score": "max",
        "policy_shock_score": "max",

        # =========================
        # Topic Metrics
        # =========================
        "topic_concentration": "mean",

        "topic_energy_supply": "max",
        "topic_fed_monetary_policy": "max",
        "topic_geopolitical_oil_risk": "max",
        "topic_policy_shock": "max",
        "topic_risk_sentiment": "max",
        "topic_trump_energy_policy": "max",
        "topic_trump_market_shock_language": "max",
        "topic_usd_strength_pressure": "max",
        "topic_wti_bearish": "max",
        "topic_wti_bullish": "max",

        # =========================
        # Semantic Change
        # =========================
        "semantic_global_novelty": "max",
        "semantic_local_change": "max",
        "semantic_rolling_novelty": "max",
    }

    # =========================
    # Embeddings
    # =========================
    for c in emb_cols:
        agg_dict[c] = "mean"

    agg = (
        df.groupby("minute")
          .agg(agg_dict)
          .reset_index()
    )

    return agg

### 5. Leakage-Free WTI Mapping

Wird ein Tweet um 12:37:21 gepostet, mappt die Funktion den WTI Kurs von 12:36:00 hinzu. Das soll Data Leakage verhindern, da wir den WTI Daten nicht entnehmen können, ob der Kurs um 12:37:00 den Kurs von 12:36:00 - 12:36:59 oder 12:37:00 bis 12:37:59 meint

In [13]:
def map_to_wti(tweet_agg, wti):

    tweet_agg = tweet_agg.copy()
    wti = wti.copy()

    tweet_agg[TIME_TWEET] = pd.to_datetime(tweet_agg["minute"], utc=True)
    wti[TIME_WTI] = pd.to_datetime(wti[TIME_WTI], utc=True)

    # KEY RULE:
    # tweet -> previous completed minute candle
    tweet_agg["wti_time"] = tweet_agg["minute"] - pd.Timedelta(minutes=1)

    df = tweet_agg.merge(
        wti,
        left_on="wti_time",
        right_on=TIME_WTI,
        how="left"
    )

    df = df.dropna(subset=["open"])

    return df

### 6. Feature / Target Split

In [14]:
def split_xy(df):

    all_targets = [
        c for c in df.columns 
        if c.startswith("y_")
    ]

    y = df[CONFIG["targets"]]

    X = df.drop(columns=all_targets)

    return X, y


# TARGETS = [c for c in wti.columns if c.startswith("y_")]

# def split_xy(df):

#     y = df[TARGETS]
#     X = df.drop(columns=TARGETS)

#     return X, y

### 7. Evaluation

In [15]:
def evaluate(y_true, prob):

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        # "f1": f1_score(y_true, pred),
        # "acc": accuracy_score(y_true, pred)
    }

### 8. Clean Features

In [16]:
def clean_features(df):

    # print(df.isna().sum()[df.isna().sum() > 0])

    # nur numerische Features
    df = df.select_dtypes(include=[np.number])

    # df = df.dropna()

    return df


### 9. Build Dataset

Datensatz wird erstellt, der später in den Training Loop geht

In [17]:
def build_dataset(wti_df, tweets_df):

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)

    return df, X, y

## 10. Feature Sets

Feature-Sets für den Baseline und den um Tweet Infos angereicherten Datensatz

In [18]:
def get_feature_sets(X):

    baseline_cols = [
        'open', 'high', 'low', 'close', 'was_missing',
        'log_close', 'ret_1', 'hl_range', 'oc_return', 'ret_5', 'ret_15',
        'ret_30', 'ret_60', 'ret_240', 'mom_5', 'mom_15', 'mom_60',
        'vol_close_5', 'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15',
        'vol_close_30', 'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60',
        'vol_close_240', 'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60',
        'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15',
        'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240',
        'hl_momentum', 'body', 'upper_wick', 'lower_wick', 'abs_ret',
        'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek', 'hour_sin',
        'hour_cos', 'dow_sin', 'dow_cos', 'us_session', 'vol_rank',
        'vol_regime', 'trend_60', 'trend_regime'
    ]

    return {
        "baseline": clean_features(X[baseline_cols].copy()),
        "enhanced": clean_features(X.copy())
    }

### 11. Time Series Split

Zeitliche Abhängigkeiten bleiben erhalten und es wird nicht geshuffled

In [19]:
def outer_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["outer_splits"],
        gap=CONFIG["gap"]
    )

def inner_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["inner_splits"],
        gap=CONFIG["gap"]
    )

### 12. Initialisierung der ML-Models

In [20]:
def build_model(name, params):

    if name == "logreg":
        return LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_estimators = 5000,
            early_stopping_rounds = 50,
            #verbose = False,
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            random_state=42,
            **params
        )

### 13. Correlation

entfernt stark korrelierende Features

In [21]:
class CorrelationFilter(BaseEstimator, TransformerMixin):

    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):

        X_df = (
            pd.DataFrame(X)
            if not isinstance(X, pd.DataFrame)
            else X
        )

        corr = X_df.corr().abs()

        upper = corr.where(
            np.triu(
                np.ones(corr.shape),
                k=1
            ).astype(bool)
        )

        self.to_drop_ = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]

        self.feature_names_in_ = np.array(X_df.columns)

        self.keep_mask_ = ~np.isin(
            self.feature_names_in_,
            self.to_drop_
        )

        return self


    def transform(self, X):

        X_df = pd.DataFrame(
            X,
            columns=self.feature_names_in_
        )

        return X_df.loc[:, self.keep_mask_].values

In [22]:
# class CorrelationFilter(BaseEstimator, TransformerMixin):

#     def __init__(self, threshold=0.95):
#         self.threshold = threshold

#     def fit(self, X, y=None):

#         # ensure DataFrame for correlation step
#         X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X

#         corr = X_df.corr().abs()

#         upper = corr.where(
#             np.triu(np.ones(corr.shape), k=1).astype(bool)
#         )

#         self.to_drop_ = [
#             col for col in upper.columns
#             if any(upper[col] > self.threshold)
#         ]

#         self.feature_names_in_ = list(X_df.columns)

#         return self

#     def transform(self, X):

#         X_df = pd.DataFrame(X, columns=getattr(self, "feature_names_in_", None))

#         X_df = X_df.drop(columns=self.to_drop_, errors="ignore")

#         return X_df

Embedding Columns

In [23]:
def get_embedding_cols(X):

    return [
        c
        for c in X.columns
        if c.startswith("emb_")
    ]


### 14. Preprocessor

Definiert Pipeline Schritte, die Embedding-Features werden PCA komprimiert und skaliert, die restlichen Features nur skaliert.

In [24]:
# feature_set, embedding_cols, feature_names

def build_preprocessor(emb_cols, non_emb_cols):

    # numeric_cols = [
    #     c for c in feature_names
    #     if c not in embedding_cols
    # ]

    embeddings_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=CONFIG["embedding_pca_components"]))
    ])

    # numeric_pipe = "passthrough"
    numeric_pipe = Pipeline([
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("embeddings", embeddings_pipe, emb_cols),
            ("numeric", numeric_pipe, non_emb_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    # optional baseline-only feature filtering
    # if feature_set == "baseline":
    #     return Pipeline([
    #         ("preprocessor", preprocessor),
    #         ("variance", VarianceThreshold(0.01)),
    #         ("corr", CorrelationFilter())
    #     ])

    # production path: still include correlation filtering (recommended)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("variance", VarianceThreshold(0.01)),
        ("corr", CorrelationFilter())
    ])

### 15. Pipeline

Führt für jedes Model die Preprocessing Schritte durch + anschließender Model fit

In [25]:
def make_pipeline(
    model_name,
    params,
    emb_cols,
    non_emb_cols
    # feature_set,
    # embedding_cols,
    # feature_names
):

    model = build_model(model_name, params)

    steps = []

    # -------------------------
    # PREPROCESSOR (UNIFIED)
    # -------------------------
    steps.append((
        "preprocessor",
        build_preprocessor(
            emb_cols,
            non_emb_cols
            # feature_set,
            # embedding_cols,
            # feature_names
        )
    ))

    # -------------------------
    # MODEL-SPECIFIC LOGIC
    # -------------------------
    # if model_name == "logreg":

    #     steps.append(("selector", logreg_selector()))
    #     #steps.append(("scaler", StandardScaler()))

    # elif model_name == "xgboost":

    #     steps.append(("selector", xgb_selector()))
    #     # no scaling needed

    # elif model_name == "mlp":

    #     steps.append(("selector", mlp_selector()))
    #     #steps.append(("scaler", StandardScaler()))

    # -------------------------
    # MODEL
    # -------------------------
    steps.append(("model", model))

    return Pipeline(steps)

In [26]:
# def make_pipeline(model_name, params):

#     model = build_model(model_name, params)

#     steps = []

#     # scaling only for linear / neural
#     if model_name in ["logreg", "mlp"]:
#         steps.append(("scaler", StandardScaler()))

#     # feature selection only for tree models
#     if model_name == "xgboost":
#         steps.append(("selector", TreeFeatureSelector(model)))

#     steps.append(("model", model))

#     return Pipeline(steps)

### 16. Feature Namen aus Pipeline extrahieren

In [27]:
def extract_final_feature_names(preprocessor):

    # nach ColumnTransformer
    names = (
        preprocessor
        .named_steps["preprocessor"]
        .get_feature_names_out()
    )


    # nach VarianceThreshold
    variance_mask = (
        preprocessor
        .named_steps["variance"]
        .get_support()
    )

    names = names[variance_mask]


    # nach CorrelationFilter
    corr_mask = (
        preprocessor
        .named_steps["corr"]
        .keep_mask_
    )

    names = names[corr_mask]


    return np.array(names)

### 17. SHAP + Importance

In [28]:
def calculate_feature_importance(
    model_name,
    model,
    X_transformed,
    feature_names
):

    results = []


    # =========================
    # LOGISTIC REGRESSION
    # =========================

    if model_name == "logreg":

        explainer = shap.LinearExplainer(
            model,
            X_transformed
        )

        shap_values = explainer(
            X_transformed
        ).values


    # =========================
    # XGBOOST
    # =========================

    elif model_name == "xgboost":

        explainer = shap.TreeExplainer(
            model
        )

        shap_values = explainer(
            X_transformed
        ).values


    # =========================
    # MLP
    # =========================

    elif model_name == "mlp":

        background = shap.sample(
            X_transformed,
            200,
            random_state=42
        )


        explainer = shap.Explainer(
            model.predict_proba,
            background
        )

        shap_values = explainer(
            X_transformed
        ).values



    # =========================
    # GLOBAL IMPORTANCE
    # =========================

    # mean_abs_shap = (
    #     np.abs(shap_values)
    #     .mean(axis=0)
    # )
    
    mean_abs_shap = np.abs(shap_values)

    if mean_abs_shap.ndim == 3:
        # (samples, features, classes)
        mean_abs_shap = mean_abs_shap.mean(axis=(0, 2))

    elif mean_abs_shap.ndim == 2:
        # (samples, features)
        mean_abs_shap = mean_abs_shap.mean(axis=0)

    else:
        raise ValueError(
            f"Unexpected SHAP shape: {mean_abs_shap.shape}"
        )
    
    if len(feature_names) != len(mean_abs_shap):
        raise ValueError(
            f"Feature mismatch: "
            f"{len(feature_names)=}, "
            f"{len(mean_abs_shap)=}, "
            f"{np.shape(shap_values)=}"
        )
    
    importance = pd.DataFrame({

        "feature":
            feature_names,

        "mean_abs_shap":
            mean_abs_shap

    })


    importance = (
        importance
        .sort_values(
            "mean_abs_shap",
            ascending=False
        )
    )


    return importance, shap_values

In [29]:
def extract_horizon_minutes(target_name):

    match = re.search(r"y_up_(\d+)", target_name)

    if match:
        return int(match.group(1))

    return None

### 18. Optuna - Model Tuning

- automatische Hyperparameter-Optimierung mittels Optuna
- Features werden nicht preselcted und nur über die Regularisierungs-Parameter "entfernt" (alos Gewicht = 0)

Optuna (inner CV only)

In [30]:
def tune_model(model_name, X_train, y_train, emb_cols, non_emb_cols, inner):

    def objective(trial):

        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-4, 1e2, log=True),  # enger & stärker regularisiert # vorher war 1.0 upper bound
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)  # wenn solver elasticnet
            }

        elif model_name == "xgboost":
            params = {
                # "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            
                "max_depth": trial.suggest_int("max_depth", 2, 6),   # <<< KEY CHANGE
            
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            
                "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),

                # 🔥 STRONG regularization (currently missing)
                "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
            
                "gamma": trial.suggest_float("gamma", 0, 5),
            
                "reg_lambda": trial.suggest_float("reg_lambda", 1, 20),
            
                "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),            
            }

        elif model_name == "mlp":
            h1 = trial.suggest_int("h1", 16, 64)
            h2 = trial.suggest_int("h2", 8, 32)

            params = {
                "hidden_layer_sizes": (h1, h2),

                "alpha": trial.suggest_float("alpha", 1e-4, 1e-1, log=True),

                "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 5e-3, log=True),

                "batch_size": trial.suggest_categorical("batch_size",[32, 64, 128, 256]),

                "activation": trial.suggest_categorical("activation", ["relu", "tanh"])
            }

        scores = []
        # inner = inner_cv()

        for tr, va in inner.split(X_train):

            X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
            y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]
            
            if model_name == "xgboost":
                # -------------------------
                # Train -> Train_ES Split
                # -------------------------
                es_size = int(len(X_tr) * 0.2)

                X_fit = X_tr.iloc[:-es_size]
                y_fit = y_tr.iloc[:-es_size]

                X_es = X_tr.iloc[-es_size:]
                y_es = y_tr.iloc[-es_size:]

                # -------------------------
                # Preprocessing
                # -------------------------
                preprocessor = build_preprocessor(emb_cols, non_emb_cols)

                X_fit_prep = preprocessor.fit_transform(X_fit, y_fit)

                X_es_prep = preprocessor.transform(X_es)

                X_va_prep = preprocessor.transform(X_va)

                # -------------------------
                # Model
                # -------------------------
                model = build_model(model_name, params)

                model.fit(
                    X_fit_prep,
                    y_fit,
                    eval_set=[(X_es_prep, y_es)],
                    verbose=False
                )

                prob = model.predict_proba(X_va_prep)[:, 1]
                                
            else:
                pipe = make_pipeline(model_name, params, emb_cols, non_emb_cols)
                pipe.fit(X_tr, y_tr)

                prob = pipe.predict_proba(X_va)[:, 1]
            scores.append(roc_auc_score(y_va, prob))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=CONFIG["optuna_trials"])

    best_params = study.best_params

    if model_name == "mlp":
        best_params = best_params.copy()
        best_params["hidden_layer_sizes"] = (
            best_params.pop("h1"),
            best_params.pop("h2")
        )


    return best_params

### 19. Finaler Model Fit

Hier wird das jeweilige ML-Model mit den optimierten Hyperparmaetern auf den CV-Fold gefittet und validiert

Outer Fold Training (Unbiased)

In [31]:
def run_fold(model_name, best_params, X_train, X_val, y_train, y_val, emb_cols, non_emb_cols):

    if model_name == "xgboost":
        # -------------------------
        # Train -> Train_ES Split
        # -------------------------
        es_size = int(len(X_train) * 0.2)

        X_fit = X_train.iloc[:-es_size]
        y_fit = y_train.iloc[:-es_size]

        X_es = X_train.iloc[-es_size:]
        y_es = y_train.iloc[-es_size:]

        # -------------------------
        # Preprocessing
        # -------------------------
        preprocessor = build_preprocessor(emb_cols, non_emb_cols)

        X_fit_prep = preprocessor.fit_transform(X_fit, y_fit)

        X_es_prep = preprocessor.transform(X_es)

        X_va_prep = preprocessor.transform(X_val)

        X_train_prep = preprocessor.transform(X_train)

        # -------------------------
        # Model
        # -------------------------
        pipe = build_model(model_name, best_params)

        pipe.fit(
            X_fit_prep,
            y_fit,
            eval_set=[(X_es_prep, y_es)],
            verbose=False
        )

        prob_train = pipe.predict_proba(X_train_prep)[:, 1]
        prob_val = pipe.predict_proba(X_va_prep)[:, 1]

        return pipe, preprocessor, prob_train, prob_val, X_train_prep, X_va_prep

    else:

        pipe = make_pipeline(model_name, best_params, emb_cols, non_emb_cols)

        pipe.fit(X_train, y_train)

        prob_train = pipe.predict_proba(X_train)[:, 1]
        prob_val = pipe.predict_proba(X_val)[:, 1]

        # pred_train = (prob_train > 0.5).astype(int)
        # pred_val = (prob_val > 0.5).astype(int)

        return pipe, pipe.named_steps["preprocessor"], prob_train, prob_val, pipe.named_steps["preprocessor"].transform(X_train), pipe.named_steps["preprocessor"].transform(X_val)

### 20. Experiment Loop (Full nested CV)

In dieser Training Loop werden die Schritte 1-19 automatisiert für die Kombinationen aus Target, Feature-Set und ML-Model durchgeführt und die Ergebnisse + Modellkonfigurationen weggeschrieben

In [32]:
def run_experiment(wti_df, tweets_df):

    df, X, y = build_dataset(wti_df, tweets_df)
    feature_sets = get_feature_sets(X)

    outer = outer_cv()
    inner = inner_cv()

    results = []
    predictions = []
    models = {}

    for target in CONFIG["targets"]:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():
            emb_cols = [c for c in X_fs.columns if c.startswith("emb_")]
            non_emb_cols = [c for c in X_fs.columns if c not in emb_cols]
            
            for model_name in CONFIG["models"]:

                # -------------------------
                # OUTER CV (EVALUATION)
                # -------------------------
                for fold, (tr, va) in enumerate(outer.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # -------------------------
                    # INNER CV (TUNING)
                    # -------------------------
                    best_params = tune_model(
                        model_name,
                        X_train,
                        y_train,
                        emb_cols,
                        non_emb_cols,
                        inner
                    )

                    model, preprocessor, prob_train, prob_val, X_train_prep, X_val_prep = run_fold(
                        model_name,
                        best_params,
                        X_train,
                        X_val,
                        y_train,
                        y_val,
                        emb_cols,
                        non_emb_cols
                    )

                    #scores_train = evaluate(y_train, prob_train)
                    #scores_val = evaluate(y_val, prob_val)

                    scores_train = roc_auc_score(y_train, prob_train)
                    scores_val = roc_auc_score(y_val, prob_val)

                    feature_names = extract_final_feature_names(
                        preprocessor
                    )
                    importance, shap_values = calculate_feature_importance(
                        model_name, 
                        model.named_steps["model"] 
                        if model_name != "xgboost" 
                        else model, 
                        X_val_prep, 
                        feature_names
                    )

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        "train_auc": scores_train,
                        "val_auc": scores_val,
                        "y_train_true": y_train.values,
                        "y_train_prob": prob_train,
                        "y_train_index": y_train.index,
                        "y_val_true": y_val.values,
                        "y_val_prob": prob_val,
                        "y_val_index": y_val.index,
                        "params": best_params,
                        "features": feature_names.tolist(),
                        "feature_importance": importance.to_dict("records"),
                        "shap_values": shap_values,
                        "feature_timestamp": df.loc[y_val.index,"minute"].values,
                        "target_timestamp": df.loc[y_val.index,"minute"].values + pd.Timedelta(minutes=extract_horizon_minutes(target))
                    })

                    # predictions.append({
                    #     "target": target,
                    #     "feature_set": fs_name,
                    #     "model": model_name,
                    #     "fold": fold,
                    #     "y_true": y_val.values,
                    #     "y_pred": pred,
                    #     "y_prob": prob,
                    #     "index": y_val.index
                    # })

                    models[(target, fs_name, model_name, fold)] = {"model": model, "preprocessor": preprocessor}

    return results, models

## 21. Function Call: Training Loop

In [33]:
results, models = run_experiment(wti, tweets_pre2020)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_21644\2612417994.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
[I 2026-06-17 11:57:56,344] A new study created in memory with name: no-name-22f13612-64a2-4049-8ca6-74d017558319
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\sit

In [34]:
results_df = pd.DataFrame(results)

#results_df.to_pickle(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results_final.pkl")


## Ergebnisse

In [35]:
results_df.head()

,target,feature_set,model,fold,train_auc,val_auc,y_train_true,y_train_prob,y_train_index,y_val_true,y_val_prob,y_val_index,params,features,feature_importance,shap_values,feature_timestamp,target_timestamp
0,y_up_2,baseline,logreg,0,0.754734,0.759482,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.518106411746593, 0.48276264132020574, 0.696...","RangeIndex(start=0, stop=3031, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0916633401169435, 0.07252778941748349, 0.07...","RangeIndex(start=3031, stop=6061, step=1)","{'C': 0.02202560662987694, 'l1_ratio': 0.59777...","[open, was_missing, ret_1, hl_range, oc_return...","[{'feature': 'dayofweek', 'mean_abs_shap': 0.4...","[[0.0, -0.2609569805518673, -0.0, 0.0, 0.00195...","[2018-05-25T23:13:00.000000, 2018-05-26T00:37:...","[2018-05-25T23:15:00.000000, 2018-05-26T00:39:..."
1,y_up_2,baseline,logreg,1,0.762095,0.751321,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.5604515525629119, 0.4537667262594586, 0.679...","RangeIndex(start=0, stop=6061, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.6173753976959621, 0.533694283073928, 0.6218...","RangeIndex(start=6061, stop=9091, step=1)","{'C': 0.8245126555372012, 'l1_ratio': 0.612864...","[open, was_missing, ret_1, hl_range, oc_return...","[{'feature': 'dayofweek', 'mean_abs_shap': 0.8...","[[-0.08074657315093584, 0.15060516439455987, 0...","[2019-04-26T12:39:00.000000, 2019-04-26T12:47:...","[2019-04-26T12:41:00.000000, 2019-04-26T12:49:..."
2,y_up_2,enhanced,logreg,0,0.745970,0.753182,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.482779384358146, 0.4955385487135546, 0.5183...","RangeIndex(start=0, stop=3031, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.47516371116489514, 0.47516371116489514, 0.4...","RangeIndex(start=3031, stop=6061, step=1)","{'C': 0.0006274924096952539, 'l1_ratio': 0.321...","[pca0, pca1, pca2, pca3, pca4, pca5, pca6, pca...","[{'feature': 'vol_close_60', 'mean_abs_shap': ...","[[-0.0, -0.0, -0.0, 0.0, -0.0, 0.0, 0.0, -0.0,...","[2018-05-25T23:13:00.000000, 2018-05-26T00:37:...","[2018-05-25T23:15:00.000000, 2018-05-26T00:39:..."
3,y_up_2,enhanced,logreg,1,0.751022,0.748212,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.43706327391602445, 0.5028649737950777, 0.53...","RangeIndex(start=0, stop=6061, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5331392947399928, 0.5279941202882728, 0.554...","RangeIndex(start=6061, stop=9091, step=1)","{'C': 0.0005456727425892126, 'l1_ratio': 0.456...","[pca0, pca1, pca2, pca3, pca4, pca5, pca6, pca...","[{'feature': 'was_missing', 'mean_abs_shap': 0...","[[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0...","[2019-04-26T12:39:00.000000, 2019-04-26T12:47:...","[2019-04-26T12:41:00.000000, 2019-04-26T12:49:..."


In [36]:
#joblib.dump(models, r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\models_final.joblib")

#models_loaded = joblib.load(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\models_final.joblib")

In [37]:
#results_df = pd.read_pickle(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results_final.pkl")

In [38]:
raise SystemExit("Notebook hier stoppen")

SystemExit: Notebook hier stoppen

c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Hinweis: Prüfung auf Signifikanz

In [ ]:
def build_model(name, params):

    if name == "logreg":
        return LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_estimators = 500,
            #early_stopping_rounds = 50,
            #verbose = False,
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            random_state=42,
            **params
        )

In [ ]:
def fit_final_models(
    wti_train,
    tweets_train,
    best_params_table
):

    df, X, y = build_dataset(
        wti_train,
        tweets_train
    )

    feature_sets = get_feature_sets(X)

    final_models = {}

    for target in CONFIG["targets"]:

        y_t = y[target]


        for fs_name, X_fs in feature_sets.items():


            emb_cols = [
                c for c in X_fs.columns 
                if c.startswith("emb_")
            ]

            non_emb_cols = [
                c for c in X_fs.columns
                if c not in emb_cols
            ]


            for model_name in CONFIG["models"]:


                print(
                    target,
                    fs_name,
                    model_name
                )


                params = best_params_table[
                    (
                        target,
                        fs_name,
                        model_name
                    )
                ]


                # -----------------
                # FINAL FIT
                # -----------------

                model = make_pipeline(
                    model_name,
                    params,
                    emb_cols,
                    non_emb_cols
                )


                model.fit(
                    X_fs,
                    y_t
                )


                final_models[
                    (
                        target,
                        fs_name,
                        model_name
                    )
                ] = model


    return final_models

In [ ]:
best_params_table = {}

for _,r in results_df.iterrows():

    key = (
        r["target"],
        r["feature_set"],
        r["model"]
    )

    if key not in best_params_table:

        best_params_table[key] = r["params"]

In [ ]:
best_params_table[
    ("y_up_15","enhanced","xgboost")
]

{'max_depth': 3,
 'learning_rate': 0.029320216478487694,
 'subsample': 0.7049529493808617,
 'colsample_bytree': 0.6978241563064111,
 'min_child_weight': 4.781747502599239,
 'gamma': 1.0357172706386368,
 'reg_lambda': 12.412559022657588,
 'reg_alpha': 2.913320085264675}

In [ ]:
tweets_pre2020.shape

(9831, 53)

In [ ]:
final_models = fit_final_models(
    wti,
    tweets_pre2020,
    best_params_table
)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_29548\2612417994.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


y_up_2 baseline logreg


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


y_up_2 baseline xgboost
y_up_2 baseline mlp
y_up_2 enhanced logreg


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


...

y_up_720 enhanced xgboost
y_up_720 enhanced mlp
y_up_1440 baseline logreg


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


y_up_1440 baseline xgboost
y_up_1440 baseline mlp
y_up_1440 enhanced logreg


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


y_up_1440 enhanced xgboost
y_up_1440 enhanced mlp


In [ ]:
import joblib

joblib.dump(
    final_models,
    r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\final_models_train_full.joblib"
)

['C:\\Users\\02_Florian_Benutzer\\Desktop\\trump_oil_analysis_v2\\results\\final_models_train_full.joblib']

## Schritt 1: Test Dataset bauen

In [ ]:
df_test, X_test, y_test = build_dataset(
    wti,
    tweets_post2020
)


feature_sets_test = get_feature_sets(
    X_test
)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_29548\2612417994.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


## Schritt 2: Test Predicitons erzeugen

In [ ]:
test_predictions = []


for target in CONFIG["targets"]:

    y_true = y_test[target]


    for fs_name, X_fs in feature_sets_test.items():


        for model_name in CONFIG["models"]:


            key = (
                target,
                fs_name,
                model_name
            )


            model = final_models[key]


            prob = model.predict_proba(
                X_fs
            )[:,1]


            test_predictions.append({

                "target":target,

                "feature_set":fs_name,

                "model":model_name,


                "y_true":y_true.values,

                "prob":prob,


                "timestamp":
                    df_test.loc[
                        y_true.index,
                        "minute"
                    ].values

            })



test_predictions = pd.DataFrame(
    test_predictions
)

In [ ]:
test_predictions.head()

,target,feature_set,model,y_true,prob,timestamp
0,y_up_2,baseline,logreg,"[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, ...","[0.17523541919170052, 0.16342501502919196, 0.1...","[2020-01-01T00:55:00.000000, 2020-01-01T01:03:..."
1,y_up_2,baseline,xgboost,"[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, ...","[0.017090779, 0.017090779, 0.017090779, 0.0170...","[2020-01-01T00:55:00.000000, 2020-01-01T01:03:..."
2,y_up_2,baseline,mlp,"[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, ...","[0.11626550321833229, 0.11021102662955472, 0.1...","[2020-01-01T00:55:00.000000, 2020-01-01T01:03:..."
3,y_up_2,enhanced,logreg,"[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, ...","[0.1382779666466542, 0.15347107164790294, 0.16...","[2020-01-01T00:55:00.000000, 2020-01-01T01:03:..."
4,y_up_2,enhanced,xgboost,"[0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, ...","[0.006991589, 0.007902613, 0.0075329896, 0.010...","[2020-01-01T00:55:00.000000, 2020-01-01T01:03:..."


## Schritt 3: AUC Tabelle

In [ ]:
from sklearn.metrics import roc_auc_score


auc_results=[]


for _,row in test_predictions.iterrows():


    auc_results.append({

        "target":
            row["target"],

        "feature_set":
            row["feature_set"],

        "model":
            row["model"],


        "auc":
            roc_auc_score(
                row["y_true"],
                row["prob"]
            )

    })


auc_df = pd.DataFrame(
    auc_results
)


auc_df

,target,feature_set,model,auc
0,y_up_2,baseline,logreg,0.720626
1,y_up_2,baseline,xgboost,0.755636
2,y_up_2,baseline,mlp,0.728901
3,y_up_2,enhanced,logreg,0.717263
4,y_up_2,enhanced,xgboost,0.750746
5,y_up_2,enhanced,mlp,0.733837
6,y_up_5,baseline,logreg,0.708313
7,y_up_5,baseline,xgboost,0.749706
8,y_up_5,baseline,mlp,0.724919
9,y_up_5,enhanced,logreg,0.707668


## Schritt 4: Bootstrap AUC CI

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score


def bootstrap_auc_ci(
    y_true,
    y_prob,
    n_bootstrap=2000
):

    rng = np.random.RandomState(42)

    scores=[]


    for i in range(n_bootstrap):

        idx = rng.randint(
            0,
            len(y_true),
            len(y_true)
        )


        if len(
            np.unique(y_true[idx])
        ) < 2:

            continue


        scores.append(
            roc_auc_score(
                y_true[idx],
                y_prob[idx]
            )
        )


    return (
        roc_auc_score(
            y_true,
            y_prob
        ),

        np.percentile(
            scores,
            2.5
        ),

        np.percentile(
            scores,
            97.5
        )
    )

# Schritt 5: CI für alle Modelle

In [ ]:
ci_results=[]


for _,row in test_predictions.iterrows():


    auc, low, high = bootstrap_auc_ci(
        row["y_true"],
        row["prob"]
    )


    ci_results.append({

        "target":
            row["target"],

        "feature_set":
            row["feature_set"],

        "model":
            row["model"],


        "auc":
            auc,

        "ci_lower":
            low,

        "ci_upper":
            high

    })


ci_df = pd.DataFrame(
    ci_results
)


ci_df

,target,feature_set,model,auc,ci_lower,ci_upper
0,y_up_2,baseline,logreg,0.720626,0.704693,0.736735
1,y_up_2,baseline,xgboost,0.755636,0.742838,0.769515
2,y_up_2,baseline,mlp,0.728901,0.714444,0.743095
3,y_up_2,enhanced,logreg,0.717263,0.701689,0.733211
4,y_up_2,enhanced,xgboost,0.750746,0.736792,0.764408
5,y_up_2,enhanced,mlp,0.733837,0.719324,0.748187
6,y_up_5,baseline,logreg,0.708313,0.692416,0.723419
7,y_up_5,baseline,xgboost,0.749706,0.736142,0.763331
8,y_up_5,baseline,mlp,0.724919,0.710133,0.739388
9,y_up_5,enhanced,logreg,0.707668,0.691686,0.724159


## Schritt 6: Delta AUC Bootsrap

In [ ]:
def bootstrap_delta_auc(
    y,
    pred_base,
    pred_enh,
    n_bootstrap=2000
):

    rng=np.random.RandomState(42)

    deltas=[]


    for i in range(n_bootstrap):

        idx=rng.randint(
            0,
            len(y),
            len(y)
        )


        if len(
            np.unique(y[idx])
        ) < 2:

            continue


        auc_base = roc_auc_score(
            y[idx],
            pred_base[idx]
        )


        auc_enh = roc_auc_score(
            y[idx],
            pred_enh[idx]
        )


        deltas.append(
            auc_enh-auc_base
        )


    return np.percentile(
        deltas,
        [2.5,97.5]
    )

In [ ]:
delta_results=[]


for target in CONFIG["targets"]:

    for model in CONFIG["models"]:


        base = test_predictions[
            (test_predictions.target==target)
            &
            (test_predictions.model==model)
            &
            (test_predictions.feature_set=="baseline")
        ].iloc[0]


        enh = test_predictions[
            (test_predictions.target==target)
            &
            (test_predictions.model==model)
            &
            (test_predictions.feature_set=="enhanced")
        ].iloc[0]



        delta = (
            roc_auc_score(
                enh.y_true,
                enh.prob
            )
            -
            roc_auc_score(
                base.y_true,
                base.prob
            )
        )



        ci = bootstrap_delta_auc(
            enh.y_true,
            base.prob,
            enh.prob
        )



        delta_results.append({

            "target":target,

            "model":model,

            "delta_auc":delta,

            "delta_ci_low":ci[0],

            "delta_ci_high":ci[1]

        })


delta_df=pd.DataFrame(
    delta_results
)


delta_df

,target,model,delta_auc,delta_ci_low,delta_ci_high
0,y_up_2,logreg,-0.003363,-0.009292,0.002201
1,y_up_2,xgboost,-0.004890,-0.015755,0.005432
2,y_up_2,mlp,0.004935,-0.010173,0.020013
3,y_up_5,logreg,-0.000645,-0.005213,0.003875
4,y_up_5,xgboost,-0.001068,-0.010123,0.007792
5,y_up_5,mlp,0.002312,-0.011476,0.015546
6,y_up_15,logreg,-0.003963,-0.010056,0.002016
7,y_up_15,xgboost,0.008636,-0.002191,0.019105
8,y_up_15,mlp,-0.010604,-0.024247,0.003624
9,y_up_30,logreg,-0.002016,-0.007005,0.003182


## Schritt 7: DeLong Test

In [ ]:
import numpy as np
from scipy import stats


def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]

    N = len(x)
    T = np.zeros(N)

    i = 0

    while i < N:
        j = i

        while j < N and Z[j] == Z[i]:
            j += 1

        T[i:j] = np.arange(i, j).mean()

        i = j

    T2 = np.empty(N)
    T2[J] = T + 1

    return T2



def compute_midrank_weight(x):
    pass



def fastDeLong(predictions_sorted_transposed, label_1_count):

    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m

    positive = predictions_sorted_transposed[:, :m]
    negative = predictions_sorted_transposed[:, m:]


    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k,m])
    ty = np.empty([k,n])
    tz = np.empty([k,m+n])


    for r in range(k):

        tx[r,:] = compute_midrank(
            positive[r,:]
        )

        ty[r,:] = compute_midrank(
            negative[r,:]
        )

        tz[r,:] = compute_midrank(
            predictions_sorted_transposed[r,:]
        )


    aucs = (
        tz[:,:m].sum(axis=1)/m/n
        -
        (m+1)/2/n
    )


    v01 = (
        tz[:,:m]
        -
        tx
    ) / n


    v10 = (
        1 -
        (tz[:,m:] - ty) / m
    )


    sx = np.cov(v01)

    sy = np.cov(v10)


    delongcov = sx/m + sy/n


    return aucs, delongcov



def calc_pvalue(aucs, sigma):

    diff = np.abs(
        np.diff(aucs)
    )

    z = (
        diff /
        np.sqrt(
            sigma[0,0]
            +
            sigma[1,1]
            -
            2*sigma[0,1]
        )
    )

    return (
        2 *
        (1 - stats.norm.cdf(z))
    )



def delong_roc_test(y_true, pred1, pred2):

    order = np.argsort(-y_true)

    y_true = y_true[order]

    preds = np.vstack([
        pred1,
        pred2
    ])

    preds = preds[:,order]


    label_1_count = int(
        y_true.sum()
    )


    aucs, covariance = fastDeLong(
        preds,
        label_1_count
    )


    return calc_pvalue(
        aucs,
        covariance
    )

In [ ]:
stat_results=[]


for target in CONFIG["targets"]:

    for model in CONFIG["models"]:


        base=test_predictions[
            (test_predictions.target==target)
            &
            (test_predictions.model==model)
            &
            (test_predictions.feature_set=="baseline")
        ].iloc[0]


        enh=test_predictions[
            (test_predictions.target==target)
            &
            (test_predictions.model==model)
            &
            (test_predictions.feature_set=="enhanced")
        ].iloc[0]



        p = delong_roc_test(
            base.y_true,
            base.prob,
            enh.prob
        )


        stat_results.append({

            "target":target,

            "model":model,

            "p_value":float(p[0])

        })


delong_df=pd.DataFrame(
    stat_results
)


delong_df

,target,model,p_value
0,y_up_2,logreg,2.265058e-01
1,y_up_2,xgboost,3.504381e-01
2,y_up_2,mlp,5.249536e-01
3,y_up_5,logreg,7.767401e-01
4,y_up_5,xgboost,8.155492e-01
5,y_up_5,mlp,7.395313e-01
6,y_up_15,logreg,1.971440e-01
7,y_up_15,xgboost,1.186815e-01
8,y_up_15,mlp,1.390314e-01
9,y_up_30,logreg,4.363775e-01


## Schritt 8: Finale Tabelle zusammenbauen

In [ ]:
final_results = (

    auc_df
    .merge(
        ci_df,
        on=[
            "target",
            "feature_set",
            "model"
        ],
        suffixes=("","_ci")
    )

)


final_results

,target,feature_set,model,auc,auc_ci,ci_lower,ci_upper
0,y_up_2,baseline,logreg,0.720626,0.720626,0.704693,0.736735
1,y_up_2,baseline,xgboost,0.755636,0.755636,0.742838,0.769515
2,y_up_2,baseline,mlp,0.728901,0.728901,0.714444,0.743095
3,y_up_2,enhanced,logreg,0.717263,0.717263,0.701689,0.733211
4,y_up_2,enhanced,xgboost,0.750746,0.750746,0.736792,0.764408
5,y_up_2,enhanced,mlp,0.733837,0.733837,0.719324,0.748187
6,y_up_5,baseline,logreg,0.708313,0.708313,0.692416,0.723419
7,y_up_5,baseline,xgboost,0.749706,0.749706,0.736142,0.763331
8,y_up_5,baseline,mlp,0.724919,0.724919,0.710133,0.739388
9,y_up_5,enhanced,logreg,0.707668,0.707668,0.691686,0.724159


In [ ]:
final_summary = (

    delta_df

    .merge(
        delong_df,
        on=[
            "target",
            "model"
        ]
    )

)


final_summary

,target,model,delta_auc,delta_ci_low,delta_ci_high,p_value
0,y_up_2,logreg,-0.003363,-0.009292,0.002201,2.265058e-01
1,y_up_2,xgboost,-0.004890,-0.015755,0.005432,3.504381e-01
2,y_up_2,mlp,0.004935,-0.010173,0.020013,5.249536e-01
3,y_up_5,logreg,-0.000645,-0.005213,0.003875,7.767401e-01
4,y_up_5,xgboost,-0.001068,-0.010123,0.007792,8.155492e-01
5,y_up_5,mlp,0.002312,-0.011476,0.015546,7.395313e-01
6,y_up_15,logreg,-0.003963,-0.010056,0.002016,1.971440e-01
7,y_up_15,xgboost,0.008636,-0.002191,0.019105,1.186815e-01
8,y_up_15,mlp,-0.010604,-0.024247,0.003624,1.390314e-01
9,y_up_30,logreg,-0.002016,-0.007005,0.003182,4.363775e-01
